In [2]:
import pandas as pd
import numpy as np

# =====================================================================
# 1. CREATE MOCK DATASETS (Simulating standard Credit Bureau structures)
# =====================================================================

# Application Records (Demographics and Financials)
app_data = {
    'ID': [5008804, 5008805, 5008806, 5008807, 5008808],
    'Gender': ['M', 'F', 'M', np.nan, 'F'],  # Contains missing values
    'Annual_Income': [427500.0, 427500.0, 112500.0, np.nan, 270000.0],  # Missing values
    'Education': ['Higher ed', 'Higher ed', 'Secondary', 'Secondary', 'Higher ed']
}
df_app = pd.DataFrame(app_data)

# Credit Bureau Monthly Records (Granular tracking)
# MONTHS_BALANCE: 0 is current month, -1 is previous month, etc.
# STATUS: 0: 1-29 days past due, 1: 30-59 days, C: paid off, X: No loan
bureau_data = {
    'ID': [5008804, 5008804, 5008804, 5008805, 5008805, 5008806, 5008806, 5008808],
    'MONTHS_BALANCE': [0, -1, -2, 0, -1, 0, -12, 0],
    'STATUS': ['C', '0', 'X', '1', '0', 'C', 'X', '5'] # 5 = write-off/bad debt
}
df_bureau = pd.DataFrame(bureau_data)

print("=== STEP 1: INITIAL MISSING DATA CHECK (Application) ===")
print(df_app.isnull().sum())
print(f"Initial Shape: {df_app.shape}\n")


# =====================================================================
# 2. FEATURE ENGINEERING ON CREDIT HISTORY (MONTHS_BALANCE & STATUS)
# =====================================================================
print("=== STEP 2: AGGREGATING CREDIT ACTIVITY AND PAYMENT BEHAVIOR ===")

# Feature A: Credit Activity Period
# Calculate total duration by finding the absolute minimum window span
credit_period = df_bureau.groupby('ID')['MONTHS_BALANCE'].agg(
    Activity_Period_Months=lambda x: abs(x.min())
).reset_index()

# Feature B: Mapping Payment Behavior Status to Risk Flag
# Interpret STATUS values to isolate critical payment behavior indicators
# Regularizing types: C, X, 0 are generally non-defaulting; 1-5 mean escalating overdue states
def classify_status(status_series):
    has_serious_overdue = status_series.isin(['2', '3', '4', '5']).any()
    has_light_overdue = status_series.isin(['1', '0']).any()
    
    if has_serious_overdue:
        return 'Serious Overdue'
    elif has_light_overdue:
        return 'Timely to Light Overdue'
    else:
        return 'No Active Risk or No Loan'

payment_behavior = df_bureau.groupby('ID')['STATUS'].agg(
    Payment_Behavior_Class=classify_status
).reset_index()

# Merge granular engineered credit features together
df_bureau_profile = pd.merge(credit_period, payment_behavior, on='ID', how='inner')
print(df_bureau_profile, "\n")


# =====================================================================
# 3. MERGING DATASETS ON UNIQUE APPLICANT ID
# =====================================================================
print("=== STEP 3: LEFT MERGING TABLES ===")
# Left join preserves all core application accounts while mapping historical profiles
df_merged = pd.merge(df_app, df_bureau_profile, on='ID', how='left')
print(f"Merged Shape: {df_merged.shape}\n")


# =====================================================================
# 4. ROBUST HANDLING OF MISSING VALUES WITH MISSING METRICS
# =====================================================================
print("=== STEP 4: DATA CLEANING POST-MERGE ===")

# Diagnostic evaluation metrics check
missing_counts = df_merged.isnull().sum()
missing_pct = df_merged.isnull().mean() * 100

print(pd.DataFrame({'Null Count': missing_counts, 'Null Percentage (%)': missing_pct}))

# Data type contextual imputation block
for col in df_merged.columns:
    if df_merged[col].isnull().sum() > 0:
        if df_merged[col].dtype == 'object':
            # Categorical text or missing history mapping gets 'Unknown' or Mode
            # If Activity or Behavior metrics are empty, it means they have no record in the credit bureau
            if col in ['Payment_Behavior_Class']:
                df_merged[col] = df_merged[col].fillna('No Bureau History')
            else:
                df_merged[col] = df_merged[col].fillna(df_merged[col].mode().iloc[0])
        else:
            # Numerical data missing mapping
            if col == 'Activity_Period_Months':
                df_merged[col] = df_merged[col].fillna(0) # No record implies 0 activity months
            else:
                df_merged[col] = df_merged[col].fillna(df_merged[col].median())

print("\n=== FINAL VERIFICATION AFTER CLEANING & MERGING ===")
print(df_merged.isnull().sum())
print("\nFinal Cleaned Dataset Sample Ready for Model Training:")
print(df_merged.to_string())


=== STEP 1: INITIAL MISSING DATA CHECK (Application) ===
ID               0
Gender           1
Annual_Income    1
Education        0
dtype: int64
Initial Shape: (5, 4)

=== STEP 2: AGGREGATING CREDIT ACTIVITY AND PAYMENT BEHAVIOR ===
        ID  Activity_Period_Months     Payment_Behavior_Class
0  5008804                       2    Timely to Light Overdue
1  5008805                       1    Timely to Light Overdue
2  5008806                      12  No Active Risk or No Loan
3  5008808                       0            Serious Overdue 

=== STEP 3: LEFT MERGING TABLES ===
Merged Shape: (5, 6)

=== STEP 4: DATA CLEANING POST-MERGE ===
                        Null Count  Null Percentage (%)
ID                               0                  0.0
Gender                           1                 20.0
Annual_Income                    1                 20.0
Education                        0                  0.0
Activity_Period_Months           1                 20.0
Payment_Behavior_Cl